In [15]:
import cv2 as cv
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from contour_detection import find_hand_contour
from feature_extraction import extract_features, create_hand_mask
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

In [16]:
RPS_DATA = Path("rps2")
X = []
y= []
for class_folder in RPS_DATA.iterdir():
    #If if class_folder is actually a folder
    if not class_folder.is_dir():
        continue
    #Get label/gesture of the images of that folder
    label = class_folder.name
    for img_path in class_folder.iterdir():
        #Read the image
        img = cv.imread(str(img_path))
        #Entire image is our region of interest
        mask = create_hand_mask(img)
        hand_contour = find_hand_contour(mask)
        features = extract_features(hand_contour,img)
        if features is None:
            continue
        X.append(features)
        y.append(label)
X= pd.DataFrame(X)
y = np.array(y)

In [17]:
X.head()

,defect_count,solidity,aspect_ratio,extent,area_ratio,hull_area_ratio,circularity,hu_log_1,hu_log_2,hu_log_3,hu_log_4,hu_log_5,hu_log_6,hu_log_7,cx_ratio,cy_ratio,max_defect_depth_ratio
0,3.0,0.642973,0.673333,0.503927,0.339311,0.527722,0.118543,0.548152,1.580612,2.924208,3.051233,6.107054,3.848096,6.323673,0.312248,0.541283,0.458555
1,1.0,0.911268,0.683761,0.679995,0.282878,0.310422,0.604339,0.723018,2.092909,3.329183,4.539573,8.484844,5.715216,8.923062,0.456934,0.648096,0.146518
2,2.0,0.842590,0.593750,0.637811,0.275761,0.327278,0.381080,0.665544,1.794015,3.379613,4.395026,8.274722,5.482589,9.419595,0.430049,0.597822,0.241684
3,4.0,0.763877,0.737069,0.534533,0.235622,0.308456,0.203423,0.665945,1.894160,3.347369,4.523871,8.471246,-6.294214,8.906413,0.537861,0.623191,0.325768
4,4.0,0.712198,0.894118,0.491710,0.317644,0.446006,0.211204,0.693767,2.366613,3.992702,4.128423,-8.937848,5.341177,8.188074,0.491201,0.601454,0.346369


In [ ]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print(label_encoder.classes_)

['paper' 'rock' 'scissors']


In [19]:
X_train, X_test, y_train, y_test = train_test_split(X,y_encoded,test_size=0.2,random_state=42,stratify=y_encoded)

In [20]:
model = XGBClassifier(
    n_estimators=900,          # more trees
    max_depth=5,               # more complex trees
    learning_rate=0.02,        # slower, more careful learning

    subsample=0.9,             # use 90% of data per tree
    colsample_bytree=0.9,      # use 90% of features per tree

    min_child_weight=1,        # allow more detailed splits
    gamma=0.0,                 # allow splits more freely

    reg_alpha=0.01,            # light L1 regularization
    reg_lambda=1.5,            # moderate L2 regularization

    eval_metric="mlogloss",
    random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test,y_pred,target_names=label_encoder.classes_))

Accuracy: 0.9333333333333333
              precision    recall  f1-score   support

       paper       0.91      0.92      0.91       180
        rock       0.94      0.93      0.93       166
    scissors       0.96      0.95      0.95       179

    accuracy                           0.93       525
   macro avg       0.93      0.93      0.93       525
weighted avg       0.93      0.93      0.93       525



In [21]:
joblib.dump(model,"xgboost_rps.pkl")
joblib.dump(label_encoder,"label_encoder.pkl")

['label_encoder.pkl']